# Marketing Campaign Performance Analysis

## Step 1: Problem Definition
This project aims to analyze the performance of various marketing campaigns across multiple brands. We will predict the expected `revenue` (Regression) and determine if a campaign will be profitable (Classification).

**Rationale:** We are tackling both Regression and Classification simultaneously because business stakeholders typically need to know both the exact expected numerical return (revenue) and a simplified binary decision metric (will this be profitable: Yes/No?).

In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.base import clone
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
import joblib

## Step 2: Data Collection
We ingest the marketing data from multiple CSV files into our database, then load it into a pandas dataframe.

**Rationale:** Using a robust relational database like PostgreSQL mirrors real-world Data Engineering workflows where data is queried from a centralized data warehouse. Pandas is used for efficient data manipulation.

In [ ]:
from dotenv import load_dotenv
from urllib.parse import quote_plus

load_dotenv()
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "")
db_password_encoded = quote_plus(db_password) if db_password else ""
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "marketing_campaign")
db_url = f"postgresql://{db_user}:{db_password_encoded}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

# Ingest data
base_dir = r"c:\Users\jegad\projects\Marketing Campaign Performance Analysis"
data_dir = os.path.join(base_dir, "data")

files = {
    'nykaa': os.path.join(data_dir, 'nykaa_campaign_data_with_nulls.csv'),
    'purplle': os.path.join(data_dir, 'purplle_campaign_data_with_nulls.csv'),
    'tira': os.path.join(data_dir, 'tira_campaign_data_with_nulls.csv')
}

dfs = []
for brand, file_path in files.items():
    if os.path.exists(file_path):
        temp_df = pd.read_csv(file_path)
        temp_df.columns = [col.lower() for col in temp_df.columns]
        if 'date' in temp_df.columns:
            temp_df = temp_df.rename(columns={'date': 'date_str'})
        temp_df['brand'] = brand
        dfs.append(temp_df)

if dfs:
    raw_df = pd.concat(dfs, ignore_index=True)
    raw_df.to_sql('raw_campaign_data', engine, if_exists='replace', index=False)
    print(f"Ingested {len(raw_df)} rows into database.")
    df = pd.read_sql("SELECT * FROM raw_campaign_data", engine)
else:
    print("Data not found. Please ensure CSVs are present.")
    df = pd.DataFrame()


## Step 3: Data Cleaning & Exploratory Data Analysis (EDA)
Handling missing values, deduplication, and visualizing distributions.

**Rationale for Imputation Strategy:** We use the `median` for missing numerical columns because it is highly robust to extreme outliers (unlike the mean). We use the `mode` for categorical columns to replace missing values with the most frequently occurring category, maintaining the natural distribution of the data.

In [ ]:
# 1. Deduplication
df = df.drop_duplicates()

# 2. Impute Categorical Columns
categorical_cols = ['campaign_type', 'target_audience', 'language', 'customer_segment', 'brand']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')
    
# 3. Impute Numerical Columns
numerical_cols = ['duration', 'impressions', 'clicks', 'leads', 'conversions', 'revenue', 'acquisition_cost', 'engagement_score', 'roi']
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# --- EDA Visualizations ---
plt.figure(figsize=(10,5))
sns.histplot(df['roi'], kde=True, bins=30)
plt.title("Distribution of ROI")
plt.xlabel("ROI")
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


## Step 4: Feature Engineering
Creating derived metrics (CTR, CPL), cyclical encoding for dates, and multi-label encoding for channels.

**Rationale:** 
- **Cyclical Encoding:** Months are cyclical (December is very close to January). Standard linear encoding (1-12) fails to capture this relationship, so we use Sine/Cosine transformations.
- **Multi-Label Encoding:** A single marketing campaign can utilize multiple channels simultaneously (e.g., Email AND Facebook). One-hot encoding a combined string fails; multi-label encoding correctly isolates the impact of each channel.

In [ ]:
# Multi-Label Encoding for channel_used
df['channel_used'] = df['channel_used'].fillna('Unknown')
channels = ['YouTube', 'Instagram', 'Google', 'WhatsApp', 'Email', 'Facebook']
for channel in channels:
    df[f'channel_{channel.lower()}'] = df['channel_used'].apply(lambda x: 1 if channel in str(x) else 0)
    
# Target Variable for Classification
df['profit_flag'] = (df['roi'] > 0).astype(int)

# Parse Date column
if 'date_str' in df.columns:
    df['date_parsed'] = pd.to_datetime(df['date_str'], format='%d-%m-%Y', errors='coerce') 
    df['month'] = df['date_parsed'].dt.month.fillna(1)
    # Cyclical Encoding for month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
else:
    df['month_sin'] = 0
    df['month_cos'] = 0

# Derived Metrics
df['ctr'] = np.where(df['impressions'] > 0, df['clicks'] / df['impressions'], 0)
df['conversion_rate'] = np.where(df['clicks'] > 0, df['conversions'] / df['clicks'], 0)
df['cpl'] = np.where(df['leads'] > 0, df['acquisition_cost'] / df['leads'], 0)

print("Feature Engineering complete.")


## Step 5: Train-Test Split
Defining feature sets and splitting the data for both Regression and Classification models.

**Rationale for 80/20 Split:** An 80/20 train-test split is an industry standard based on the Pareto principle. It provides the model with enough data to learn complex patterns while retaining a sufficiently large hold-out set to accurately evaluate generalization performance.

In [ ]:
cat_features = ['campaign_type', 'target_audience', 'language', 'customer_segment', 'brand']
num_features_base = ['duration', 'impressions', 'clicks', 'leads', 'conversions', 'engagement_score', 'month_sin', 'month_cos', 'ctr', 'conversion_rate', 'cpl']
channel_features = [col for col in df.columns if col.startswith('channel_') and col != 'channel_used']

# Preprocessors
preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_base + ['acquisition_cost']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ], remainder='passthrough'
)

preprocessor_cls = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_base + ['revenue', 'acquisition_cost']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ], remainder='passthrough'
)

# Split for Regression (Predict Revenue)
X_reg = df[cat_features + num_features_base + channel_features + ['acquisition_cost']]
y_reg = df['revenue']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Split for Classification (Predict Profit Flag)
X_cls = df[cat_features + num_features_base + channel_features + ['revenue', 'acquisition_cost']]
y_cls = df['profit_flag']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

print("Train-Test splits created successfully.")


## Step 6: Model Selection
We use **XGBoost** for both regression and classification.

**Rationale:** XGBoost is chosen because it is an extremely powerful gradient boosting algorithm that handles complex, non-linear relationships in tabular data better than linear models. 

**Handling Class Imbalance:** Instead of using external balancing libraries like SMOTE, we rely on XGBoost's native `scale_pos_weight` parameter for the classification task, which internally adjusts the weights of the minority class to ensure unbiased learning.

In [ ]:
reg_pipeline = SklearnPipeline(steps=[
    ('preprocessor', clone(preprocessor_reg)),
    ('regressor', XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1))
])

# Calculate class imbalance ratio dynamically to pass to XGBoost
positive_cases = sum(y_train_c == 1)
negative_cases = sum(y_train_c == 0)
scale_pos_weight = negative_cases / max(1, positive_cases)

cls_pipeline = SklearnPipeline(steps=[
    ('preprocessor', clone(preprocessor_cls)),
    ('classifier', XGBClassifier(n_estimators=500, max_depth=15, learning_rate=0.05, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos_weight))
])


## Step 7: Model Training
Training the pipelines on the training datasets.

**Rationale for Pipelines:** We encapsulate all preprocessing steps (scaling, encoding) alongside the model estimator inside a Pipeline. This prevents 'data leakage' by guaranteeing that scaling metrics (like mean/variance) are fitted solely on the training data, ensuring a valid evaluation later.

In [ ]:
print("Training Revenue Regressor...")
reg_pipeline.fit(X_train_r, y_train_r)

print("Training Profit Classifier...")
cls_pipeline.fit(X_train_c, y_train_c)

print("Training complete!")


## Step 8: Model Evaluation
Evaluating performance on test sets.

**Rationale for Metrics:**
- **R² (R-Squared):** Explains the proportion of variance in revenue captured by the model.
- **RMSE (Root Mean Squared Error):** Provides the error in the actual monetary scale of the target variable, making it easily interpretable for business stakeholders.
- **Classification Report:** Precision/Recall metrics are more informative than raw accuracy, especially when identifying unprofitable campaigns is as important as profitable ones.

In [ ]:
# Regression Evaluation
preds_r = reg_pipeline.predict(X_test_r)
print(f"Regression R² Score: {r2_score(y_test_r, preds_r):.4f}") 
print(f"Regression RMSE: {math.sqrt(mean_squared_error(y_test_r, preds_r)):.2f}")

# Classification Evaluation
preds_c = cls_pipeline.predict(X_test_c)
print(f"\nClassification Accuracy: {accuracy_score(y_test_c, preds_c):.4f}")
print(classification_report(y_test_c, preds_c))

# Feature Importance Visualization for XGBoost (Regression)
model = reg_pipeline.named_steps['regressor']
importance = model.feature_importances_
plt.figure(figsize=(10,5))
plt.title("XGBRegressor Feature Importances (Top 10)")
indices = np.argsort(importance)[-10:]
plt.barh(range(len(indices)), importance[indices], align='center')
plt.show()


## Step 9: Model Tuning
Using `GridSearchCV` on the Regressor pipeline to find better hyperparameters (e.g., `max_depth` and `learning_rate`).

**Rationale:** Tree-based models like XGBoost are highly sensitive to hyperparameters. `GridSearchCV` systematically searches for the optimal balance between bias and variance (e.g., tuning tree depth prevents overfitting) using cross-validation to guarantee robustness.

In [ ]:
param_grid = {
    'regressor__max_depth': [3, 6],
    'regressor__learning_rate': [0.05, 0.1]
}
print("Starting Grid Search for Regression Model...")
# cv=2 used for speed in presentation notebook
grid_search = GridSearchCV(reg_pipeline, param_grid, cv=2, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_r, y_train_r)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.4f}")

best_reg_pipeline = grid_search.best_estimator_
preds_r_tuned = best_reg_pipeline.predict(X_test_r)
print(f"Tuned Regression R² Score on Test Set: {r2_score(y_test_r, preds_r_tuned):.4f}")


## Step 10: Model Deployment
Saving the best trained models for production use.

**Rationale:** We serialize the models to disk using `joblib`. This allows the exact pipeline (including all preprocessing steps) to be loaded later in an API or web application to make live predictions on new marketing campaigns.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_reg_pipeline, 'models/revenue_regressor_tuned.joblib')
joblib.dump(cls_pipeline, 'models/profit_classifier.joblib')
print("Saved pipeline models to disk!")
